### 📘 Concepto: `@before_agent` + `RemoveMessage`
- **`@before_agent`**: convierte una función en middleware que corre **una vez, al inicio** de cada `invoke` (antes del primer paso del modelo). Compara con `before_model`, que corre antes de **cada** llamada al modelo.
- La función recibe el `state` y devuelve una **actualización** del state (un `dict`), igual que una tool con `Command(update=...)`.
- **`RemoveMessage(id=...)`**: no es un mensaje que se envíe; es una **orden de borrado**. El *reducer* de `messages` (`add_messages`) ve el `RemoveMessage` y elimina el mensaje con ese `id`.
- Aquí borra **todos los `ToolMessage`** (resultados de tools). Sirve para ahorrar tokens cuando las salidas de tools son largas… pero el modelo pierde esa información.

# 3.2 Managing Messages (adaptado a lab local)

**Qué se ve aquí:** dos middlewares para controlar el historial que llega al modelo:
- `SummarizationMiddleware`: cuando el historial pasa de un umbral (`trigger`), resume lo viejo con un modelo y conserva lo último (`keep`).
- `@before_agent` + `RemoveMessage`: borra mensajes del state (aquí, todos los `ToolMessage`) antes de que corra el agente.

**Convención de este fork:** 🔸 ORIGINAL DEL CURSO (comentada, con el motivo) → 🟢 ADAPTADO LOCAL (la que corre con Ollama). Celdas sin marca = iguales al curso.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: falta cargar local_model.py (raíz del repo) y definir el modelo local.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from dotenv import load_dotenv
#
# load_dotenv()

In [1]:
# 🟢 ADAPTADO LOCAL
# Motivo: el notebook está en notebooks/module-3/ y local_model.py en la raíz del repo.
#         MODEL reemplaza a "gpt-5-nano" en todas las celdas (gemma4: soporta tools, el más rápido del lab).
import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "local_model.py").exists())))

from dotenv import load_dotenv
load_dotenv()
from local_model import get_model, OLLAMA_URL

MODEL = get_model("gemma4:latest")

## Summarize messages

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: "gpt-5-nano" es API de pago. El resumen usaba "gpt-4o-mini" (pago) → también MODEL.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.agents import create_agent
# from langgraph.checkpoint.memory import InMemorySaver
# from langchain.agents.middleware import SummarizationMiddleware
#
# agent = create_agent(
#     model="gpt-5-nano",
#     checkpointer=InMemorySaver(),
#     middleware=[
#         SummarizationMiddleware(
#             model="gpt-4o-mini",
#             trigger=("tokens", 100),
#             keep=("messages", 1)
#         )
#     ],
# )

### 📘 Concepto: `checkpointer` y `SummarizationMiddleware`

**`checkpointer=InMemorySaver()`: la libreta de memoria del agente.**
- Después de cada paso, el agente guarda una "foto" (*checkpoint*) del state (los mensajes y los campos extra) en la libreta.
- Cada conversación tiene su página, identificada por el **`thread_id`**: `{"configurable": {"thread_id": "1"}}`.
- Si vuelves a llamar con el **mismo** `thread_id`, el agente **sigue** la conversación; con otro, empieza de cero.
- `InMemory` = la libreta está en la **RAM** del kernel; se borra al reiniciarlo. En producción: `SqliteSaver` o `PostgresSaver`, o lo maneja el servidor de LangGraph (por eso en `langgraph dev` **no** se pone checkpointer).
- Aquí hace falta porque el resumen **reescribe** el historial guardado del thread, y en el módulo 3.3 (HITL) para poder **pausar y reanudar**.

**`SummarizationMiddleware`: un "resumidor" que se mete antes de cada llamada al modelo (`before_model`).**
- `trigger=("tokens", 100)`: si el historial pasa de ~100 tokens (conteo **aproximado**), se activa. Otras formas: `("messages", 50)` o `("fraction", 0.8)` del contexto del modelo.
- `keep=("messages", 1)`: conserva intacto **el último mensaje** (la pregunta actual).
- Lo que hace: borra todo (`RemoveMessage(REMOVE_ALL_MESSAGES)`) y pone **un `HumanMessage`** que empieza con *"Here is a summary of the conversation to date"* (marcado con `lc_source: "summarization"`) + los mensajes conservados.
- `model=`: el modelo que **escribe el resumen**. Puede ser otro, más barato; aquí usamos el mismo para no cargar otro en CPU.
- Costo: **una llamada extra** al modelo (la del resumen) cada vez que se pasa el umbral.

In [2]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" es API de pago. El resumen usaba "gpt-4o-mini" (pago) → también MODEL.
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model=MODEL,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=MODEL,   # el mismo modelo ya cargado en Ollama: cambiar de modelo en CPU cuesta segundos de carga,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

### 📘 Qué esperar de la celda siguiente
Le pasamos **9 mensajes** inventados (Lunapolis). Juntos pasan de 100 tokens → el middleware los resume **antes** de que el modelo responda.

En la salida, `response["messages"]` queda con solo **3 mensajes**:
1. `[0]` el **resumen** (un `HumanMessage` generado por el middleware),
2. `[1]` la última pregunta (la que pedimos conservar con `keep`),
3. `[2]` la respuesta del modelo (`AIMessage`).

Los 8 mensajes originales **ya no existen** en el state del thread `"1"`.

In [3]:
# Sin cambios
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nTo answer a series of increasingly specific and fictional questions about the location "Lunapolis" (the capital of the moon).\n\n## SUMMARY\n\nThe conversation established several fictional facts about Lunapolis:\n1.  **Capital of the Moon:** Lunapolis.\n2.  **Weather in Lunapolis:** Clear skies, high of 120C, low of -100C.\n3.  **Population Detail:** There are 100,000 cheese miners in Lunapolis.\n4.  **Political Situation:** The cheese miners\' union is expected to strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\nNone (The session appears to be a series of Q&A exchanges based on fictional premises.)', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='004ad18c-ba98-4798-8587-ec013545a81d'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' u

### 📘 Por qué `[0]` es el resumen
`response["messages"]` es una **lista** de Python. `[0]` = el **primero** de la lista. Como el middleware puso el resumen al principio, `[0].content` es el texto del resumen. (Más abajo hay una sección completa sobre cómo leer la lista.)

In [4]:
# Sin cambios
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT

To answer a series of increasingly specific and fictional questions about the location "Lunapolis" (the capital of the moon).

## SUMMARY

The conversation established several fictional facts about Lunapolis:
1.  **Capital of the Moon:** Lunapolis.
2.  **Weather in Lunapolis:** Clear skies, high of 120C, low of -100C.
3.  **Population Detail:** There are 100,000 cheese miners in Lunapolis.
4.  **Political Situation:** The cheese miners' union is expected to strike due to dissatisfaction with the new president.

## ARTIFACTS

None

## NEXT STEPS

None (The session appears to be a series of Q&A exchanges based on fictional premises.)


## Trim/delete messages

In [5]:
# Sin cambios
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: "gpt-5-nano" es API de pago.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# agent = create_agent(
#     model="gpt-5-nano",
#     checkpointer=InMemorySaver(),
#     middleware=[trim_messages],
# )

In [6]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" es API de pago.
agent = create_agent(
    model=MODEL,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

### 📘 Qué esperar de la celda siguiente (la "trampa" didáctica)
El dato de la temperatura (`temp=42C`) **solo** está en un `ToolMessage`. El middleware lo borra **antes** de que el modelo vea la conversación → el modelo **no puede** responder la temperatura.

Además: esos `ToolMessage` no tienen un `AIMessage` con `tool_calls` que los pida. Muchas APIs **rechazan** un `ToolMessage` "huérfano"; borrarlos antes también evita ese error.

In [7]:
# Sin cambios
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='cbb1eb1e-454c-4daf-a366-15f9ea720f01'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='5744e0ab-2c8f-4341-bf05-e1f48d39cd0a', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='94b59509-fd7a-452c-8d34-756bd7bab5a8'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='b11e0d63-789c-4153-bcbe-da30e578785e', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='61aeddf3-0f53-49d3-9bb4-9b20cd65f165'),
              AIMessage(content="I'm sorry, but as an AI, I have no way of knowing the physical

In [8]:
# Sin cambios
print(response["messages"][-1].content)

I'm sorry, but as an AI, I have no way of knowing the physical temperature of your device. I can only interact with you through text.

To help you troubleshoot why your device won't turn on, could you tell me a bit more about it? For example:

1.  **What kind of device is it?** (e.g., smartphone, laptop, tablet, smart TV, etc.)
2.  **What happens when you try to turn it on?** (e.g., absolutely nothing, a flicker, a sound, a logo appears and then disappears?)
3.  **What have you tried so far?** (e.g., tried a different outlet, held down the power button for a long time?)

Once I have more details, I can give you more specific troubleshooting steps!


## 🧒 Cómo leer `response["messages"]` (índices y *slicing*)

Imagina una **fila de niños** formados para entrar al salón. Cada niño es un mensaje.
- El **primero** de la fila es el número **0** (Python empieza a contar en 0, no en 1).
- Con números **negativos** cuentas **desde el final**: `-1` es el **último**, `-2` el penúltimo.
- Con **dos puntos** (`:`) tomas un **pedazo** de la fila (*slice*): `[1:3]` = desde el 1 **hasta antes** del 3.

| Escribes | Te da | Para qué se usa en el curso |
|---|---|---|
| `msgs[0]` | el primero | el resumen (Summarization) o la primera pregunta |
| `msgs[-1]` | el último | **la respuesta final del agente** (lo más usado) |
| `msgs[-2]` | el penúltimo | el `ToolMessage` justo antes de la respuesta, o la pregunta |
| `msgs[-3]` | el antepenúltimo | el `AIMessage` que **pidió** la tool (`.tool_calls`) |
| `msgs[1:]` | todos menos el primero | saltarse el resumen o el system prompt |
| `msgs[-3:]` | los 3 últimos | ver la última pregunta → tool → respuesta |
| `len(msgs)` | cuántos hay | comprobar que se borraron o resumieron mensajes |

Y en cada mensaje: `.content` = el texto, `.type` = quién habla (`human`, `ai`, `tool`), `.tool_calls` = las tools que pidió el modelo, `.response_metadata` = datos del modelo (nombre, tiempos), `.usage_metadata` = tokens.

> ¿Por qué `[-1]` y no `[5]`? Porque el número de mensajes **cambia** (si hubo tools, si se resumió…). `[-1]` **siempre** es el último, sin importar cuántos haya.

In [9]:
# 🟢 EXTRA (aprendizaje): recorrer la lista con su posición
msgs = response["messages"]          # una lista de Python
print("¿Cuántos mensajes hay?", len(msgs))
print()
for i, m in enumerate(msgs):        # enumerate da (posición, elemento)
    negativo = i - len(msgs)        # la misma posición contada desde el final
    texto = m.content[:60].replace("\n", " ")
    print(f"[{i}] = [{negativo}]  {m.type:<6} → {texto}")

¿Cuántos mensajes hay? 6

[0] = [-6]  human  → My device won't turn on. What should I do?
[1] = [-5]  ai     → Is the device plugged in and turned on?
[2] = [-4]  human  → Yes, it's plugged in and turned on.
[3] = [-3]  ai     → Is the device showing any lights or indicators?
[4] = [-2]  human  → What's the temperature of the device?
[5] = [-1]  ai     → I'm sorry, but as an AI, I have no way of knowing the physic


In [10]:
# 🟢 EXTRA (aprendizaje): índices sueltos y pedazos (slicing)
print("Primero  [0]  :", msgs[0].type, "→", msgs[0].content[:50])
print("Último   [-1] :", msgs[-1].type, "→", msgs[-1].content[:50])
print("Penúltimo[-2] :", msgs[-2].type, "→", msgs[-2].content[:50])
print()
print("Todos menos el primero [1:] →", [m.type for m in msgs[1:]])
print("Los 2 últimos [-2:]          →", [m.type for m in msgs[-2:]])
print()
ultimo = msgs[-1]
print("Modelo que respondió:", ultimo.response_metadata.get("model_name"))
print("Tokens:", ultimo.usage_metadata)

Primero  [0]  : human → My device won't turn on. What should I do?
Último   [-1] : ai → I'm sorry, but as an AI, I have no way of knowing 
Penúltimo[-2] : human → What's the temperature of the device?

Todos menos el primero [1:] → ['ai', 'human', 'ai', 'human', 'ai']
Los 2 últimos [-2:]          → ['human', 'ai']

Modelo que respondió: gemma4:latest
Tokens: {'input_tokens': 80, 'output_tokens': 175, 'total_tokens': 255}


## 🧒 La libreta del `checkpointer` en acción
El agente de *trim* usa `InMemorySaver()`. Con `agent.get_state(config)` puedes **abrir la libreta** y ver qué quedó guardado para un `thread_id`. Fíjate que los `ToolMessage` **ya no están**: el borrado quedó guardado.

In [11]:
# 🟢 EXTRA (aprendizaje): leer lo que guardó el checkpointer para el thread "2"
estado = agent.get_state({"configurable": {"thread_id": "2"}})
guardados = estado.values["messages"]
print("Mensajes guardados en el thread 2:", len(guardados))
print("Tipos:", [m.type for m in guardados])
print("¿Quedó algún ToolMessage?", any(m.type == "tool" for m in guardados))

Mensajes guardados en el thread 2: 6
Tipos: ['human', 'ai', 'human', 'ai', 'human', 'ai']
¿Quedó algún ToolMessage? False


In [12]:
# 🟢 EXTRA (aprendizaje): mismo thread = el agente recuerda; thread nuevo = empieza de cero
seguir = agent.invoke(
    {"messages": [HumanMessage(content="Summarize in one line what we talked about.")]},
    {"configurable": {"thread_id": "2"}},
)
print("Thread 2 (recuerda):", seguir["messages"][-1].content[:200])
print("Mensajes en el thread 2 ahora:", len(seguir["messages"]))

nuevo = agent.invoke(
    {"messages": [HumanMessage(content="Summarize in one line what we talked about.")]},
    {"configurable": {"thread_id": "3"}},
)
print("\nThread 3 (nuevo):", nuevo["messages"][-1].content[:200])
print("Mensajes en el thread 3:", len(nuevo["messages"]))

Thread 2 (recuerda): We are troubleshooting why your device won't turn on, and I've asked you about its power status, lights, and general behavior so far.
Mensajes en el thread 2 ahora: 8

Thread 3 (nuevo): Please provide the conversation you would like me to summarize!
Mensajes en el thread 3: 2
